# 04 — Modélisation prédictive du churn

## Ce que je vais faire ici
On a notre table de 4 518 clients avec 12 features et une cible CHURN. 
On va entraîner **3 modèles ML** pour apprendre à prédire le churn, les 
comparer rigoureusement, et choisir le meilleur.

## Plan :
1. **Préparation** : chargement, imputation des NaN, encodage
2. **Split** train/test stratifié (80/20)
3. **3 modèles** entraînés :
   - Logistic Regression (baseline simple et interprétable)
   - Random Forest (puissant, gère les non-linéarités)
   - XGBoost (standard de l'industrie ML tabulaire)
4. **Évaluation** : ROC-AUC, PR-AUC, F1, matrice de confusion
5. **Validation croisée** sur le meilleur modèle
6. **Feature importance** : qu'est-ce qui prédit le churn ?
7. **Threshold tuning** : combien de churneurs détectés à différents seuils ?
8. **Sortie Power BI** : scores de churn par client

## Pourquoi cette étape est cruciale
La segmentation (étape 3) dit "qui sont mes clients". La modélisation dit 
"quels clients vont churner et qu'est-ce que je peux y faire". C'est la 
partie qui rend ton projet **actionnable** pour un client e-commerce.

## 1. Imports et chargement

On charge la table enrichie de l'étape 3. On garde toutes les features 
sauf `segment_rfm` et `kmeans_cluster` qui ne doivent pas servir au modèle 
(elles sont dérivées des features RFM, donc redondantes).

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import joblib

# Sklearn
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    confusion_matrix, classification_report,
    roc_curve, precision_recall_curve
)

# XGBoost
from xgboost import XGBClassifier

# Configuration
pd.set_option('display.max_columns', None)
sns.set_style("whitegrid")
np.random.seed(42)

# Chargement
DATA_PROCESSED = Path("../data/processed")
df = pd.read_csv(DATA_PROCESSED / "customer_features_segmented.csv")

print(f"Données chargées : {len(df):,} clients × {df.shape[1]} colonnes")
print(f"\nTaux de churn : {df['churn'].mean()*100:.2f}%")

Données chargées : 4,518 clients × 19 colonnes

Taux de churn : 48.92%


## 2. Préparation des features

On sépare X (les features) et y (la cible). On exclut :
- `customer_id` : identifiant, pas une feature
- `churn` : la cible elle-même
- `n_orders_future` : information du futur (data leakage)
- `segment_rfm`, `RFM_score`, `R_score`, `F_score`, `M_score`, `kmeans_cluster` : 
  dérivés des features de base

On garde nos **12 features originales**.

In [3]:
# Colonnes à exclure
cols_to_drop = [
    'customer_id', 'churn', 'n_orders_future',
    'segment_rfm', 'RFM_score', 
    'R_score', 'F_score', 'M_score', 
    'kmeans_cluster'
]

X = df.drop(columns=cols_to_drop)
y = df['churn']

print(f"✅ Features utilisées ({X.shape[1]}) :")
for col in X.columns:
    print(f"  - {col}")

print(f"\nTaille de X : {X.shape}")
print(f"Distribution de y : {y.value_counts().to_dict()}")

✅ Features utilisées (10) :
  - recency
  - frequency
  - monetary
  - avg_basket
  - std_basket
  - n_distinct_products
  - tenure_days
  - avg_interpurchase_days
  - n_cancellations
  - cancellation_rate

Taille de X : (4518, 10)
Distribution de y : {0: 2308, 1: 2210}


## 3. Imputation des NaN

On a 31% de NaN sur `std_basket` et `avg_interpurchase_days` (les 
mono-acheteurs).

**Stratégie** :
- `std_basket = 0` → cohérent (un client avec 1 commande a variance 0)
- `avg_interpurchase_days = max observé` → traduit "intervalle infini"

XGBoost gère nativement les NaN, mais Logistic Regression et Random Forest 
non. Donc on impute pour avoir un dataset propre pour les 3 modèles.

In [4]:
# Imputation
X['std_basket'] = X['std_basket'].fillna(0)
X['avg_interpurchase_days'] = X['avg_interpurchase_days'].fillna(X['avg_interpurchase_days'].max())

print(f"✅ NaN restants : {X.isna().sum().sum()}")
print(f"   (devrait être 0)")

✅ NaN restants : 0
   (devrait être 0)


## 4. Split train/test stratifié

On sépare 80% pour l'entraînement, 20% pour le test final.

**`stratify=y`** garantit que le ratio churn/actif est identique dans 
les deux ensembles. Sans ça, on pourrait avoir 60% de churn dans train 
et 40% dans test, ce qui fausserait l'évaluation.

**`random_state=42`** garantit que le split est reproductible. Si tu 
relances le notebook, tu auras exactement le même train/test.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.20, 
    random_state=42, 
    stratify=y
)

print(f" Train : {len(X_train):,} clients ({y_train.mean()*100:.1f}% churn)")
print(f" Test  : {len(X_test):,} clients ({y_test.mean()*100:.1f}% churn)")

 Train : 3,614 clients (48.9% churn)
 Test  : 904 clients (48.9% churn)


## 5. Standardisation (pour Logistic Regression uniquement)

La Logistic Regression est **sensible à l'échelle** des features. Une 
feature en milliers (Monetary) et une feature entre 0 et 1 
(cancellation_rate) ne doivent pas dominer le modèle juste parce que 
leurs valeurs sont plus grandes.

On standardise donc (moyenne 0, écart-type 1) pour la Logistic. Random 
Forest et XGBoost n'en ont pas besoin (insensibles à l'échelle).

**Important** : on **fit** le scaler sur le train uniquement, puis on 
transforme train ET test. Sinon, fitter sur test = data leakage.

In [9]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Données standardisées pour Logistic Regression")
print(f"   Moyenne train ≈ 0 : {X_train_scaled.mean():.4f}")
print(f"   Écart-type train ≈ 1 : {X_train_scaled.std():.4f}")

Données standardisées pour Logistic Regression
   Moyenne train ≈ 0 : 0.0000
   Écart-type train ≈ 1 : 1.0000


## 6. Modèle 1 — Logistic Regression (baseline)

C'est notre **point de référence**. Modèle linéaire simple. Si nos 
modèles plus sophistiqués ne le battent pas, c'est qu'on a un problème.

### Paramètres
- `max_iter=1000` : laisse au solver le temps de converger
- `random_state=42` : reproductibilité

On entraîne, on prédit sur le test, et on regarde les performances.

In [10]:
# Entraînement
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)

# Prédictions
y_pred_lr = lr_model.predict(X_test_scaled)
y_proba_lr = lr_model.predict_proba(X_test_scaled)[:, 1]

# Métriques
print("=== Logistic Regression — Performances ===")
print(f"ROC-AUC  : {roc_auc_score(y_test, y_proba_lr):.4f}")
print(f"PR-AUC   : {average_precision_score(y_test, y_proba_lr):.4f}")
print(f"F1-score : {f1_score(y_test, y_pred_lr):.4f}")
print(f"\nMatrice de confusion :")
print(confusion_matrix(y_test, y_pred_lr))

=== Logistic Regression — Performances ===
ROC-AUC  : 0.8071
PR-AUC   : 0.7837
F1-score : 0.7229

Matrice de confusion :
[[344 118]
 [125 317]]


## 7. Modèle 2 — Random Forest

Modèle d'**ensemble** : 200 arbres de décision qui votent. Capture 
naturellement les non-linéarités et les interactions entre features.

### Paramètres
- `n_estimators=200` : 200 arbres (bon compromis vitesse/performance)
- `max_depth=10` : profondeur max par arbre (évite l'overfitting)
- `random_state=42` : reproductibilité
- `n_jobs=-1` : parallélise sur tous les cœurs CPU

Pas besoin de standardiser pour Random Forest.

In [11]:
rf_model = RandomForestClassifier(
    n_estimators=200, 
    max_depth=10,
    random_state=42, 
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)
y_proba_rf = rf_model.predict_proba(X_test)[:, 1]

print("=== Random Forest — Performances ===")
print(f"ROC-AUC  : {roc_auc_score(y_test, y_proba_rf):.4f}")
print(f"PR-AUC   : {average_precision_score(y_test, y_proba_rf):.4f}")
print(f"F1-score : {f1_score(y_test, y_pred_rf):.4f}")
print(f"\nMatrice de confusion :")
print(confusion_matrix(y_test, y_pred_rf))

=== Random Forest — Performances ===
ROC-AUC  : 0.8092
PR-AUC   : 0.7702
F1-score : 0.7256

Matrice de confusion :
[[335 127]
 [118 324]]


## 8. Modèle 3 — XGBoost

Le **standard de l'industrie** pour le ML tabulaire. Gradient boosting 
optimisé. Souvent le meilleur sur ce type de problème.

### Paramètres
- `n_estimators=200` : 200 itérations de boosting
- `max_depth=6` : profondeur des arbres (plus petite que RF car boostés)
- `learning_rate=0.1` : standard
- `eval_metric='logloss'` : métrique d'optimisation interne

In [12]:
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss',
    use_label_encoder=False
)
xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)
y_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]

print("=== XGBoost — Performances ===")
print(f"ROC-AUC  : {roc_auc_score(y_test, y_proba_xgb):.4f}")
print(f"PR-AUC   : {average_precision_score(y_test, y_proba_xgb):.4f}")
print(f"F1-score : {f1_score(y_test, y_pred_xgb):.4f}")
print(f"\nMatrice de confusion :")
print(confusion_matrix(y_test, y_pred_xgb))

c:\Users\nakhi\anaconda3\Lib\site-packages\xgboost\training.py:200: UserWarning: [01:51:49] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


=== XGBoost — Performances ===
ROC-AUC  : 0.8021
PR-AUC   : 0.7684
F1-score : 0.7229

Matrice de confusion :
[[344 118]
 [125 317]]
